In [1]:
from google.colab import drive
drive.mount('/content/drive')

# ── Copy source code ───────────────────────────────────────────────────────
!cp /content/drive/MyDrive/Uzcosmos_Project/src.zip /content/
!unzip -qo src.zip

# ── LEVIR-CD ───────────────────────────────────────────────────────────────
!cp /content/drive/MyDrive/Uzcosmos_Project/levir_train.zip /content/
!cp /content/drive/MyDrive/Uzcosmos_Project/levir_val.zip   /content/
!mkdir -p data/ready/levir/train data/ready/levir/val
!unzip -qo levir_train.zip -d data/ready/levir/train
!unzip -qo levir_val.zip   -d data/ready/levir/val

# ── Init Python packages ───────────────────────────────────────────────────
!touch src/__init__.py src/data/__init__.py src/models/__init__.py src/training/__init__.py src/inference/__init__.py src/evaluation/__init__.py src/utils/__init__.py

Mounted at /content/drive


In [2]:
!pip install pytorch-lightning albumentations timm opencv-python-headless torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 47.7 MB/s eta 0:00:00


In [7]:
import os
import cv2
from pathlib import Path
from tqdm import tqdm

SRC = Path("/content/data/ready/levir")
DST = Path("/content/data/ready/levir_256")

PATCH = 256
STRIDE = 256

for split in ["train", "val"]:
    for sub in ["A", "B", "label"]:
        (DST / split / sub).mkdir(parents=True, exist_ok=True)

def patchify_split(split):
    A_dir = SRC / split / "A"
    B_dir = SRC / split / "B"
    L_dir = SRC / split / "label"

    A_paths = sorted(A_dir.glob("*.png"))

    count = 0
    for a_path in tqdm(A_paths, desc=f"Patchifying {split}"):
        name = a_path.stem
        b_path = B_dir / f"{name}.png"
        l_path = L_dir / f"{name}.png"

        A = cv2.imread(str(a_path), cv2.IMREAD_COLOR)
        B = cv2.imread(str(b_path), cv2.IMREAD_COLOR)
        L = cv2.imread(str(l_path), cv2.IMREAD_GRAYSCALE)

        if A is None or B is None or L is None:
            print("Missing:", a_path, b_path, l_path)
            continue

        h, w = L.shape

        for y in range(0, h - PATCH + 1, STRIDE):
            for x in range(0, w - PATCH + 1, STRIDE):
                A_patch = A[y:y+PATCH, x:x+PATCH]
                B_patch = B[y:y+PATCH, x:x+PATCH]
                L_patch = L[y:y+PATCH, x:x+PATCH]

                out_name = f"{name}_{y}_{x}.png"

                cv2.imwrite(str(DST / split / "A" / out_name), A_patch)
                cv2.imwrite(str(DST / split / "B" / out_name), B_patch)
                cv2.imwrite(str(DST / split / "label" / out_name), L_patch)

                count += 1

    print(split, "patches:", count)

patchify_split("train")
patchify_split("val")

Patchifying train: 100%|██████████| 445/445 [01:51<00:00,  3.99it/s]


train patches: 7120


Patchifying val: 100%|██████████| 64/64 [00:15<00:00,  4.22it/s]

val patches: 1024


In [8]:
from glob import glob
import cv2, numpy as np

for split in ["train", "val"]:
    A = glob(f"/content/data/ready/levir_256/{split}/A/*.png")
    B = glob(f"/content/data/ready/levir_256/{split}/B/*.png")
    L = glob(f"/content/data/ready/levir_256/{split}/label/*.png")

    print(split)
    print("A:", len(A), "B:", len(B), "L:", len(L))

    ratios = []
    for p in L[:2000]:
        m = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        ratios.append((m > 127).mean())

    print("mean changed ratio:", np.mean(ratios))
    print("zero masks:", sum(r == 0 for r in ratios), "/", len(ratios))

train
A: 7120 B: 7120 L: 7120
mean changed ratio: 0.044588272094726565
zero masks: 1114 / 2000
val
A: 1024 B: 1024 L: 1024
mean changed ratio: 0.04196566343307495
zero masks: 588 / 1024


In [10]:
%env PYTHONPATH=/content

!python src/training/train.py \
  --model custom \
  --data_dirs /content/data/ready/levir_256 \
  --epochs 5 \
  --batch_size 8 \
  --patience 5 \
  --checkpoint_dir /content/drive/MyDrive/Uzcosmos_Project/checkpoints/debug_custom_256_no_hnm/

Streaming output truncated to the last 5000 lines.
                                                               0.924 val_recall:
                                                               0.816            
                                                               val_best_f1_cali…
                                                               0.857 val_obj_f1:
                                                               0.603            
                                                               train_loss_epoch:
                                                               39.723 train_f1: 
                                                               0.785 train_mf1: 
                                                               0.888            
                                                               train_recall:    
Epoch 4/4  ━━━━━━━━━━━━╸━━━ 702/890 0:03:44 • 0:01:00 3.17it/s v_num: 0.000     
                                                          

In [11]:
%env PYTHONPATH=/content

!python src/training/train.py \
  --model custom \
  --data_dirs /content/data/ready/levir_256 \
  --epochs 5 \
  --batch_size 8 \
  --patience 5 \
  --hard_negative_mining \
  --checkpoint_dir /content/drive/MyDrive/Uzcosmos_Project/checkpoints/debug_custom_256_hnm/

Streaming output truncated to the last 5000 lines.
                                                               0.917 val_recall:
                                                               0.854            
                                                               val_best_f1_cali…
                                                               0.842 val_obj_f1:
                                                               0.627            
                                                               train_loss_epoch:
                                                               27.307 train_f1: 
                                                               0.740 train_mf1: 
                                                               0.864            
                                                               train_recall:    
Epoch 4/4  ━━━━━━━━━━━━╸━━━ 701/890 0:03:31 • 0:00:55 3.44it/s v_num: 0.000     
                                                          

In [12]:
#checking flags exist or not
!python src/training/train.py --help | grep -E "no_tcdm|no_cscp|no_lgc|no_build_heads|hard_negative"

                [--hard_negative_mining] [--no_tcdm] [--no_cscp] [--no_bgr]
                [--no_lgc] [--no_build_heads]
  --hard_negative_mining
  --no_tcdm             Replace TCDM with simple abs-diff (ablation)
  --no_cscp             Disable cross-scale context propagation (ablation)
  --no_lgc              Disable Lightweight Global Context (ablation)
  --no_build_heads      Disable per-timestep building aux heads (ablation)


In [13]:
!cd /content/data/ready && zip -r /content/drive/MyDrive/Uzcosmos_Project/levir_256.zip levir_256

Streaming output truncated to the last 5000 lines.
  adding: levir_256/train/A/train_109_256_512.png (deflated 1%)
  adding: levir_256/train/A/train_146_512_0.png (deflated 2%)
  adding: levir_256/train/A/train_357_256_768.png (deflated 1%)
  adding: levir_256/train/A/train_142_256_0.png (deflated 1%)
  adding: levir_256/train/A/train_363_256_0.png (deflated 0%)
  adding: levir_256/train/A/train_336_768_0.png (deflated 7%)
  adding: levir_256/train/A/train_170_256_512.png (deflated 13%)
  adding: levir_256/train/A/train_411_256_768.png (deflated 18%)
  adding: levir_256/train/A/train_114_256_256.png (deflated 0%)
  adding: levir_256/train/A/train_146_0_768.png (deflated 1%)
  adding: levir_256/train/A/train_37_512_256.png (deflated 7%)
  adding: levir_256/train/A/train_80_0_256.png (deflated 1%)
  adding: levir_256/train/A/train_221_256_512.png (deflated 11%)
  adding: levir_256/train/A/train_324_768_768.png (deflated 12%)
  adding: levir_256/train/A/train_394_768_768.png (deflated 1%)